# Notebook 19: CP Violation — Structure or Data?

**Status:** v1.1 (final)
**Project:** QCA Structure & Data

## Goal
Determine whether **CP violation** can be derived from the QCA structure, or whether it is a contingent input parameter (data).

## Theoretical Framework
By CPT theorem (verified in Notebook 12), CP violation is equivalent to T violation.
A naive complex mass in 1D Dirac or 2-generation models does **not** produce physical CP violation, because the phase can be removed by a chiral rotation (an unphysical redefinition of the fields).

Physical CP violation requires at least one of the following:
1. **≥ 3 Dirac generations** (proven structurally valid in Notebook 18).
2. **Majorana masses** (neutrino sector).
3. **Topological \(\theta\)-term**.

## Strategy & Tests

We test the following cases (T0–T5):

| ID | Configuration | Expected | Purpose |
|----|---------------|----------|---------|
| **T0** | 1D Dirac, real \(m\) | CP ✓ | Baseline |
| **T1** | 1D Dirac, complex mass (chiral rotation) | CP ✓ (phase unphysical) | Negative control |
| **T2** | 2 Dirac generations, complex \(2\times2\) \(M\) | CP ✓ (no physical phase) | Negative control |
| **T4** | Toy \(3\times3\) mass matrix with phase \(\delta\) | CP breaks (\(J \neq 0\)) | Positive control |
| **T5** | 2 Majorana generations (neutrino sector) | CP may break | Majorana check |

## Metrics

For the operator-level symmetry, we compute:
\[
 \epsilon_{CP} = \frac{\| \mathcal{CP} \, H \, \mathcal{CP}^{-1} - H \|_F}{\|H\|_F}
\]
where \(\mathcal{CP}\) acts as \(\gamma_0 K\) (matrix multiplication followed by complex conjugation).

For data-level symmetry in the 3-generation case, we compute the **Jarlskog invariant**:
\[
 J = \mathrm{Im}\left( M_{11} M_{22} M_{12}^* M_{21}^* \right)
\]

## Expected Verdict
**CP violation = DATA.**
The QCA framework *can host* CP violation (via 3 generations or Majorana), but the specific value of the CP-violating phase \(\delta\) is not determined by the QCA rules. It must be given as an input parameter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 5)})

# Pauli matrices / Матрицы Паули
s1 = np.array([[0, 1], [1, 0]])
s2 = np.array([[0, -1j], [1j, 0]])
s3 = np.diag([1, -1])

# CP transformation for 1D Dirac: H -> gamma_0 * H^* * gamma_0^{-1}
# CP-преобразование для 1D Дирака: H -> gamma_0 * H^* * gamma_0^{-1}
def cp_transform(H):
    gamma_0 = s3
    return gamma_0 @ np.conj(H) @ np.linalg.inv(gamma_0)

def cp_residual(H):
    H_cp = cp_transform(H)
    return np.linalg.norm(H_cp - H) / np.linalg.norm(H)

In [ ]:
# ============================================================
# T0 & T1: 1D Dirac with real and complex mass
# T0 и T1: 1D Дирак с реальной и комплексной массой
# ============================================================
def H_1d(m_real, m_imag=0.0, k=0.0):
    """1D Dirac Hamiltonian: H = k * s1 + m_real * s3 + m_imag * s2"""
    return k * s1 + m_real * s3 + m_imag * s2

# T0: Real mass / Реальная масса
H_T0 = H_1d(m_real=0.5)
res_T0 = cp_residual(H_T0)

# T1: Complex mass, equivalent to m * exp(i * theta * gamma_5)
# T1: Комплексная масса, эквивалентна m * exp(i * theta * gamma_5)
H_T1 = H_1d(m_real=0.5, m_imag=0.3)
res_T1 = cp_residual(H_T1)

print("T0 (real mass):    CP residual = {:.2e}  (expected ~0)".format(res_T0))
print("T1 (complex mass): CP residual = {:.2e}  (expected ~0, phase unphysical)".format(res_T1))

# Spectrum check / Проверка спектра
evals_T0 = np.linalg.eigvalsh(H_T0)
evals_T1 = np.linalg.eigvalsh(H_T1)
m_mag_T1 = np.sqrt(0.5**2 + 0.3**2)
print(f"T0 eigenvalues: {evals_T0}  (|m| = 0.5000)")
print(f"T1 eigenvalues: {evals_T1}  (|m| = {m_mag_T1:.4f}; CP residual = 0, phase unphysical)")

In [ ]:
# ============================================================
# T2: 2 Dirac generations, complex 2x2 mass matrix
# T2: 2 поколения Дирака, комплексная матрица масс 2x2
# ============================================================
# For 2 generations, the most general mass matrix is 2x2.
# However, the phase can be removed by bi-unitary transformations.
# Для 2 поколений наиболее общая матрица масс — 2x2.
# Однако фаза устраняется биунитарным преобразованием.

M_complex = np.array([[1.0, 0.5 * np.exp(1j * 0.3)],
                      [0.5 * np.exp(-1j * 0.3), 2.0]])

# Singular values (physical masses) / Сингулярные числа (физические массы)
masses = la.svdvals(M_complex)
print(f"T2: Masses from 2x2 complex matrix: {masses}")

# Compare with real matrix (no phase) / Сравнение с реальной матрицей (без фазы)
M_real = np.array([[1.0, 0.5], [0.5, 2.0]])
masses_0 = la.svdvals(M_real)
print(f"T2: Masses without phase:          {masses_0}")
print("T2: Identical -> no physical CP violation in 2 generations.")

In [ ]:
# ============================================================
# T4: 3 generations — CP violation in the CKM matrix
# T4: 3 поколения — CP-нарушение в матрице CKM
# ============================================================
# CP violation requires TWO mass matrices (up and down).
# We construct a toy CKM matrix parameterized by a single phase delta.
# CP-нарушение требует ДВЕ матрицы масс (up и down).
# Мы строим toy-матрицу CKM, параметризованную одной фазой delta.

def jarlskog_from_ckm(V):
    """J = Im(V_us V_cb V_ub^* V_cs^*)"""
    return np.imag(V[0,1] * V[1,2] * np.conj(V[0,2]) * np.conj(V[1,1]))

def build_ckm(delta):
    """Build a toy CKM matrix parameterized by a single phase delta."""
    # Standard parameterization with theta_12, theta_23, theta_13
    th12, th23, th13 = 0.22, 0.04, 0.003
    c12, s12 = np.cos(th12), np.sin(th12)
    c23, s23 = np.cos(th23), np.sin(th23)
    c13, s13 = np.cos(th13), np.sin(th13)
    e_id = np.exp(1j * delta)
    e_mid = np.exp(-1j * delta)

    V = np.array([
        [c12*c13, s12*c13, s13*e_mid],
        [-s12*c23 - c12*s23*s13*e_id, c12*c23 - s12*s23*s13*e_id, s23*c13],
        [s12*s23 - c12*c23*s13*e_id, -c12*s23 - s12*c23*s13*e_id, c23*c13]
    ])
    return V

delta = np.pi / 3
V = build_ckm(delta)
J = jarlskog_from_ckm(V)

print(f"T4: CKM matrix with delta = {delta:.3f} rad")
print(f"T4: Jarlskog invariant J = {J:.4e}")
print(f"T4: CP violated? {abs(J) > 1e-10}")

# Check delta = 0 / Проверка delta = 0
V_real = build_ckm(0.0)
J_real = jarlskog_from_ckm(V_real)
print(f"T4: J with delta = 0: {J_real:.4e} (expected ~0)")

In [ ]:
# ============================================================
# T5: 2 Majorana generations (neutrino sector)
# T5: 2 майорановских поколения (нейтринный сектор)
# ============================================================
# In the Majorana case, the mass matrix is symmetric (M = M^T).
# A 2x2 Majorana matrix can have 2 physical phases.
# В майорановском случае матрица масс симметрична (M = M^T).
# 2x2 майорановская матрица может иметь 2 физические фазы.

M_maj = np.array([[1.0, 0.5 * np.exp(1j * 0.4)],
                  [0.5 * np.exp(1j * 0.4), 2.0]])

# For Majorana, CP violation is measured by Im(M11 * M22 * (M12^*)^2)
# Для майорановского случая CP-нарушение измеряется Im(M11 * M22 * (M12^*)^2)
J_maj = np.imag(M_maj[0,0] * M_maj[1,1] * np.conj(M_maj[0,1])**2)
print(f"T5: Majorana phase = 0.4 rad")
print(f"T5: Majorana CP invariant = {J_maj:.4e}")
print(f"T5: CP violated in neutrino sector? {abs(J_maj) > 1e-10}")

In [ ]:
# ============================================================
# Plot: Jarlskog invariant vs CP phase
# График: инвариант Ярлскога vs CP-фаза
# ============================================================
deltas = np.linspace(0, 2*np.pi, 200)
J_vals = [jarlskog_from_ckm(build_ckm(d)) for d in deltas]

fig_19, ax = plt.subplots(figsize=(8, 5))
ax.plot(deltas / np.pi, J_vals, color='purple', lw=2)
ax.set_xlabel(r'CP phase $\delta / \pi$')
ax.set_ylabel(r'Jarlskog invariant $J$')
ax.set_title(r'CP violation in 3 generations: $J(\delta)$')
ax.axhline(0, color='k', ls=':', lw=1)
ax.axvline(0.5, color='r', ls='--', lw=1, label=r'$\delta = \pi/2$')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig_19.savefig("19_cp_violation.png", dpi=150, bbox_inches="tight")
print("Saved: 19_cp_violation.png")

In [ ]:
# ============================================================
# Summary — Final Verdict
# Итог — финальный вердикт
# ============================================================
print("=" * 60)
print("Notebook 19 — CP VIOLATION VERDICT")
print("=" * 60)
print()
print("T0 (1D, real m):        CP conserved (residual ~0)")
print("T1 (1D, complex m):     CP conserved (phase unphysical)")
print("T2 (2 generations):     CP conserved (no physical phase)")
print("T4 (3 generations, CKM): CP violated (J != 0)")
print("T5 (Majorana, 2 gen):   CP violated (Majorana phase)")
print()
print("FINAL: CP violation is DATA.")
print("The QCA framework can host CP violation (via 3 generations")
print("or Majorana masses), but the specific phase delta is not")
print("derived from QCA rules. It is a contingent input parameter.")
print("=" * 60)